# 체성분 수치 → 헬스도감 운동 루틴 추천 (Main Quest 2 PoC, 2단계)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jsseo0420/ai-agent-mainquest2-inbody-routine-poc/blob/main/notebooks/routine.ipynb)

1단계(`poc.ipynb`)에서 Gemini Flash가 결과지 사진으로 추출한 수치를 받아, **헬스도감에 있는 운동만으로** 주 3회 루틴을 만든다.

| 방식 | 설명 |
|---|---|
| 기준선: 범용 챗봇식 | 수치만 주고 "헬스장 루틴을 짜 줘"라고 묻는다. 운동 이름은 자유롭게 쓴다 |
| 제안 방식 | ① 코드가 수치를 기준표로 판정 → ② LLM이 **도감 운동 ID 목록 안에서** 루틴을 JSON으로 구성 → ③ 코드가 도감 밖 ID를 한 번 더 지움 |

**평가 항목**: 추천 운동의 도감 포함률, 루틴 규칙(하루 4~6개, 2~4세트, 8~15회, 가슴·등·하체 포함 등) 준수율, 목표 반영

### 실행 전 준비 (Colab)
- GPU가 필요 없다. 기본 런타임(CPU)으로 실행한다.
- 왼쪽 **🔑 Secrets**에 `GEMINI_API_KEY`를 넣고 **노트북 액세스**를 켠다.
- 무료 등급 한도(분당 5회) 때문에 호출 사이에 13초씩 쉰다. 15명 × 2방식 = 30회라 10분 정도 걸린다.
- 결과는 `results/`에 저장되어, 다시 실행하면 실패한 것만 다시 부른다.

> 결과지와 수치는 모두 가상 데이터이며, 추천 결과는 의료 조언이 아니다.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/jsseo0420/ai-agent-mainquest2-inbody-routine-poc.git"
if not os.path.exists("data/ground_truth.json"):
    if not os.path.exists("poc_repo"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "poc_repo"], check=True)
    os.chdir("poc_repo")
print("작업 폴더:", os.getcwd())

!pip install -q google-genai pandas

In [ ]:
import json, re, time
from pathlib import Path
from typing import Optional

import pandas as pd


def get_secret(name):
    """Colab Secrets → 환경변수 순서로 키를 찾는다. 값은 출력하지 않는다."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Colab Secrets에 GEMINI_API_KEY를 넣고 노트북 액세스를 켜 주세요"

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
FORCE_RERUN = False  # True면 저장된 결과를 무시하고 다시 호출한다

## 1. 입력 데이터

- **수치**: 1단계에서 Gemini Flash가 추출한 값 (`results/B_gemini_flash.json`). 15장 모두 정답과 같았다.
- **성별**: 결과지에서 뽑는 8개 필드에는 없고, 앱에서는 사용자가 직접 고른다. 여기서는 생성기의 정답 값을 쓴다.
- **운동 목록**: 헬스도감의 운동 20개 (`data/exercises.json`)

In [ ]:
truth = json.loads(Path("data/ground_truth.json").read_text(encoding="utf-8"))
SAMPLES = truth["samples"]
FIELD_NAMES = truth["fields"]
extracted = {r["file"]: r["pred"] for r in json.loads(Path("results/B_gemini_flash.json").read_text(encoding="utf-8"))}

catalog = json.loads(Path("data/exercises.json").read_text(encoding="utf-8"))
EXERCISES = {e["id"]: e for e in catalog["exercises"]}
MUSCLES = {m["id"]: m for m in catalog["muscles"]}

PROFILES = [{"file": s["file"], "sex": s["sex"], "values": extracted[s["file"]]} for s in SAMPLES]
print(f"사용자 {len(PROFILES)}명, 도감 운동 {len(EXERCISES)}개")

## 2. 수치 판정 (코드)

체지방률·근육량·BMI·복부지방률이 표준보다 높은지 낮은지는 **LLM이 아니라 코드가** 기준표로 정한다.
같은 수치면 언제나 같은 판정이 나오게 하기 위해서다. 기준은 `prompts/routine_guidelines.md`에 있다 (PoC용 단순 기준).

In [ ]:
def assess(values, sex):
    """수치와 성별로 상태를 판정하고 목표를 정한다. 값이 없으면 '알 수 없음'."""
    male = sex == "M"
    w, smm, pbf, bmi, whr = (values.get(k) for k in ["weight_kg", "skeletal_muscle_kg", "body_fat_pct", "bmi", "whr"])
    lo, hi = (10, 20) if male else (18, 28)
    fat = "알 수 없음" if pbf is None else "낮음" if pbf < lo else "높음" if pbf > hi else "표준"
    bmi_s = "알 수 없음" if bmi is None else "낮음" if bmi < 18.5 else "높음" if bmi >= 25 else "표준"
    whr_s = "알 수 없음" if whr is None else "높음" if whr > (0.90 if male else 0.85) else "표준"
    muscle = "알 수 없음" if not (w and smm) else "낮음" if smm / w < (0.40 if male else 0.34) else "표준"
    if fat == "높음" and muscle == "낮음":
        goal = "근육량 늘리기 + 체지방 줄이기"
    elif fat == "높음":
        goal = "체지방 줄이기"
    elif muscle == "낮음":
        goal = "근육량 늘리기"
    else:
        goal = "지금 몸 유지하며 고르게 강화"
    return {"체지방률": fat, "근육량": muscle, "BMI": bmi_s, "복부지방률": whr_s, "목표": goal}


for p in PROFILES:
    p["status"] = assess(p["values"], p["sex"])
pd.DataFrame([{"파일": p["file"], "성별": p["sex"], **p["status"]} for p in PROFILES]).set_index("파일")

## 3. Gemini 호출 (한도에 맞춰 천천히, 실패하면 다시 시도)

In [ ]:
from google import genai
from google.genai import types
from google.genai import errors as genai_errors
from pydantic import BaseModel

GEMINI_MODEL = "gemini-3.5-flash-lite"  # gemini-3.5-flash는 혼잡해 루틴 하나에 234초가 걸려 lite로 평가했다. 바꾸면 results의 routine_*.json을 지우고 다시 실행한다
MIN_INTERVAL = 13  # 초. 무료 등급 분당 5회 한도
MAX_TRIES = 5

gemini = genai.Client(api_key=GEMINI_API_KEY)
_last_call = [0.0]


def call_gemini(prompt, schema):
    """429·503이면 기다렸다가 다시 시도한다. (파싱된 JSON, 실제 응답 시간)을 돌려준다."""
    for attempt in range(1, MAX_TRIES + 1):
        wait = MIN_INTERVAL - (time.monotonic() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.monotonic()
        start = time.perf_counter()
        try:
            resp = gemini.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=schema),
            )
            return json.loads(resp.text), round(time.perf_counter() - start, 2)
        except genai_errors.APIError as e:
            if e.code not in (429, 503) or attempt == MAX_TRIES:
                raise
            m = re.search(r"retry in ([\d.]+)s", str(e))
            delay = float(m.group(1)) + 1 if m else 15 * attempt
            print(f"    {e.code} → {delay:.0f}초 기다렸다가 다시 시도 ({attempt}/{MAX_TRIES})")
            time.sleep(delay)


def run_all(name, make_prompt, schema):
    """사용자마다 한 번씩 호출하고 results/{name}.json에 저장한다. 성공한 결과는 다시 부르지 않는다."""
    cache = RESULTS / f"{name}.json"
    saved = {} if FORCE_RERUN or not cache.exists() else {
        r["file"]: r for r in json.loads(cache.read_text(encoding="utf-8")) if r["error"] is None}
    out = []
    for p in PROFILES:
        if p["file"] in saved:
            out.append(saved[p["file"]])
            continue
        try:
            data, seconds = call_gemini(make_prompt(p), schema)
            rec = {"file": p["file"], "output": data, "seconds": seconds, "error": None}
        except Exception as e:
            rec = {"file": p["file"], "output": None, "seconds": None, "error": f"{type(e).__name__}: {e}"[:300]}
        out.append(rec)
        print(f"[{name}] {p['file']}: {'성공' if rec['error'] is None else '실패 ' + rec['error']}  {rec['seconds']}s")
        cache.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
    cache.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
    return out


def values_text(p):
    sex = "남성" if p["sex"] == "M" else "여성"
    vals = "\n".join(f"- {FIELD_NAMES[k]}: {v}" for k, v in p["values"].items())
    return f"성별: {sex}\n{vals}"

## 4. 기준선: 범용 챗봇식 추천

사용자가 ChatGPT 같은 챗봇에 수치를 붙여 넣고 물어보는 상황을 흉내 낸다. 운동 목록을 주지 않고, 이름도 자유롭게 쓰게 한다.

In [ ]:
class FreeExercise(BaseModel):
    name: str
    sets: int
    reps: str


class FreeDay(BaseModel):
    title: str
    exercises: list[FreeExercise]


class FreeRoutine(BaseModel):
    days: list[FreeDay]


def baseline_prompt(p):
    return f"""헬스장에 막 등록한 입문자의 체성분 측정 결과야. 주 3회 헬스장에서 할 운동 루틴을 짜 줘.
운동 이름은 한국어로 써 줘.

{values_text(p)}"""


baseline = run_all("routine_baseline", baseline_prompt, FreeRoutine)

## 5. 제안 방식: 판정 결과 + 도감 운동 목록 안에서 추천

- 루틴 작성 규칙은 `prompts/routine_guidelines.md`의 **"루틴 작성 규칙"** 부분을 그대로 넣는다.
- 운동은 ID로만 고르게 하고, 각 운동의 주·보조 자극 부위를 함께 알려 준다.
- 응답을 받은 뒤 **도감에 없는 ID는 코드에서 지운다** (`filter_routine`).

In [ ]:
class RoutineExercise(BaseModel):
    id: str
    sets: int
    reps: str
    reason: str


class RoutineDay(BaseModel):
    title: str
    exercises: list[RoutineExercise]


class Routine(BaseModel):
    summary: str
    days: list[RoutineDay]
    cardio_note: str
    caution: str


guide_text = Path("prompts/routine_guidelines.md").read_text(encoding="utf-8")
RULES = guide_text.split("## 루틴 작성 규칙")[1].split("\n", 1)[1].strip()


def muscle_names(ids):
    return ", ".join(MUSCLES[m]["name"] for m in ids) or "없음"


CATALOG_TEXT = "\n".join(
    f"- {e['id']}: {e['name_ko']} (주: {muscle_names(e['primary_muscles'])} / 보조: {muscle_names(e['secondary_muscles'])})"
    for e in EXERCISES.values()
)


def ours_prompt(p):
    status = ", ".join(f"{k} {v}" for k, v in p["status"].items() if k != "목표")
    return f"""너는 헬스 입문자에게 헬스도감 앱의 운동으로 주 3회 루틴을 짜 주는 트레이너다.

## 사용자
{values_text(p)}
- 코드 판정: {status}
- 목표: {p['status']['목표']}

## 규칙
{RULES}

## 헬스도감 운동 목록 (이 ID만 쓴다)
{CATALOG_TEXT}

summary에는 판정 결과와 목표를 입문자가 이해할 수 있게 2~3문장으로 쓴다. 모든 문장은 한국어 해요체로 쓴다."""


def filter_routine(routine):
    """도감에 없는 운동 ID를 지운다. (정리된 루틴, 지운 ID 목록)을 돌려준다."""
    removed = []
    days = []
    for day in routine["days"]:
        kept = []
        for ex in day["exercises"]:
            (kept if ex["id"] in EXERCISES else removed).append(ex if ex["id"] in EXERCISES else ex["id"])
        days.append({**day, "exercises": kept})
    return {**routine, "days": days}, removed


ours = run_all("routine_ours", ours_prompt, Routine)

## 6. 평가

### 도감 포함률
- 제안 방식: LLM이 낸 ID가 도감에 있는지 그대로 센다 (**필터 전**). 필터 후는 정의상 100%다.
- 기준선: 자유롭게 쓴 운동 이름을 도감 운동 이름과 맞춰 본다. 띄어쓰기·"머신"을 빼고 이름이 들어 있으면 포함으로 센다.
  예를 들어 "덤벨 숄더 프레스"도 "숄더 프레스"가 들어 있어 포함으로 세므로, **기준선의 실제 포함률은 이보다 낮다** (기준선에 유리한 채점).

In [ ]:
def norm(s):
    s = re.sub(r"\(.*?\)", "", s.lower())
    return re.sub(r"[\s·\-_/]|머신|machine", "", s)


# 도감 운동마다 이름 키를 만든다. 긴 키부터 맞춰 "인클라인 벤치프레스"가 "벤치프레스"로 잘못 잡히지 않게 한다.
ALIASES = {
    "flat-bench-press": ["벤치프레스", "benchpress"],
    "seated-cable-row": ["시티드로우", "케이블로우"],
    "assisted-pullup-dip": ["풀업", "딥스", "pullup", "dip"],
    "back-extension": ["백익스텐션"],
    "hip-abduction-adduction": ["어브덕션", "어덕션"],
    "ab-crunch": ["크런치"],
    "smith-machine": ["스미스"],
    "pec-deck": ["펙덱"],
}
NAME_KEYS = sorted(
    [(norm(part), e["id"]) for e in EXERCISES.values() for part in re.sub(r"\s벤치$", "", e["name_ko"]).split("/") + [e["name_en"]]]
    + [(norm(a), i) for i, names in ALIASES.items() for a in names],
    key=lambda kv: -len(kv[0]),
)


def match_catalog(name):
    n = norm(name)
    return next((i for key, i in NAME_KEYS if key and key in n), None)


def rep_numbers(reps):
    return [int(x) for x in re.findall(r"\d+", reps)]


def check_ours(rec, p):
    r = rec["output"]
    ids = [ex["id"] for d in r["days"] for ex in d["exercises"]]
    filtered, removed = filter_routine(r)
    groups = {MUSCLES[m]["group"] for i in ids if i in EXERCISES for m in EXERCISES[i]["primary_muscles"]}
    legs = any(m in ("quads", "glutes") for i in ids if i in EXERCISES for m in EXERCISES[i]["primary_muscles"])
    days = filtered["days"]
    checks = {
        "3일 구성": len(days) == 3,
        "하루 4~6개": all(4 <= len(d["exercises"]) <= 6 for d in days),
        "같은 날 중복 없음": all(len({e["id"] for e in d["exercises"]}) == len(d["exercises"]) for d in days),
        "2~4세트": all(2 <= e["sets"] <= 4 for d in days for e in d["exercises"]),
        "8~15회": all(rep_numbers(e["reps"]) and all(8 <= n <= 15 for n in rep_numbers(e["reps"])) for d in days for e in d["exercises"]),
        "가슴·등·하체 포함": "chest" in groups and "back" in groups and legs,
        "체지방 높음 → 유산소 안내": p["status"]["체지방률"] != "높음" or bool(r["cardio_note"].strip()),
    }
    return {
        "운동 수": len(ids),
        "도감 포함(필터 전)": sum(i in EXERCISES for i in ids),
        "지운 ID": ", ".join(removed),
        **checks,
    }


def check_baseline(rec):
    names = [ex["name"] for d in rec["output"]["days"] for ex in d["exercises"]]
    matched = [match_catalog(n) for n in names]
    return {
        "운동 수": len(names),
        "도감 포함": sum(m is not None for m in matched),
        "도감 밖 운동": ", ".join(n for n, m in zip(names, matched) if m is None),
    }


prof = {p["file"]: p for p in PROFILES}
ours_eval = pd.DataFrame([{"파일": r["file"], **check_ours(r, prof[r["file"]])} for r in ours if r["error"] is None]).set_index("파일")
base_eval = pd.DataFrame([{"파일": r["file"], **check_baseline(r)} for r in baseline if r["error"] is None]).set_index("파일")
ours_eval.to_csv(RESULTS / "routine_ours_eval.csv", encoding="utf-8-sig")
base_eval.to_csv(RESULTS / "routine_baseline_eval.csv", encoding="utf-8-sig")

rule_cols = [c for c in ours_eval.columns if ours_eval[c].dtype == bool]
summary = pd.DataFrame([
    {"방식": "기준선 (범용 챗봇식)", "성공한 호출": f"{len(base_eval)}/{len(baseline)}",
     "도감 포함률": f"{base_eval['도감 포함'].sum() / base_eval['운동 수'].sum():.1%}",
     "도감 밖 운동이 있는 루틴": f"{(base_eval['도감 포함'] < base_eval['운동 수']).sum()}/{len(base_eval)}",
     "평균 응답 시간(초)": round(pd.Series([r["seconds"] for r in baseline if r["seconds"]]).mean(), 1)},
    {"방식": "제안 방식 (필터 전)", "성공한 호출": f"{len(ours_eval)}/{len(ours)}",
     "도감 포함률": f"{ours_eval['도감 포함(필터 전)'].sum() / ours_eval['운동 수'].sum():.1%}",
     "도감 밖 운동이 있는 루틴": f"{(ours_eval['도감 포함(필터 전)'] < ours_eval['운동 수']).sum()}/{len(ours_eval)}",
     "평균 응답 시간(초)": round(pd.Series([r["seconds"] for r in ours if r["seconds"]]).mean(), 1)},
    {"방식": "제안 방식 (필터 후)", "성공한 호출": f"{len(ours_eval)}/{len(ours)}", "도감 포함률": "100.0%",
     "도감 밖 운동이 있는 루틴": f"0/{len(ours_eval)}", "평균 응답 시간(초)": None},
]).set_index("방식")
summary.to_csv(RESULTS / "routine_summary.csv", encoding="utf-8-sig")
summary

In [ ]:
# 제안 방식의 루틴 규칙 준수율 (필터 후 기준)
rules = (ours_eval[rule_cols].mean()).to_frame("준수율")
rules.to_csv(RESULTS / "routine_rules.csv", encoding="utf-8-sig")
rules.style.format("{:.0%}")

In [ ]:
# 기준선이 추천한 도감 밖 운동 (자주 나온 순)
outside = pd.Series([n.strip() for s in base_eval["도감 밖 운동"] for n in s.split(",") if n.strip()])
outside.value_counts().head(15)

In [ ]:
# 규칙을 어긴 루틴 (실패 사례 후보)
ours_eval[~ours_eval[rule_cols].all(axis=1)]

## 7. 루틴 예시

첫 번째 사용자의 루틴을 헬스도감 링크와 함께 보여 준다. 앱 화면도 이 구조를 그대로 쓴다.

In [ ]:
from IPython.display import Markdown, display


def show_routine(file):
    p = prof[file]
    rec = next(r for r in ours if r["file"] == file)
    routine, removed = filter_routine(rec["output"])
    lines = [f"### {file} · {'남성' if p['sex'] == 'M' else '여성'} · 목표: {p['status']['목표']}", "", routine["summary"], ""]
    for d in routine["days"]:
        lines.append(f"**{d['title']}**")
        for e in d["exercises"]:
            ex = EXERCISES[e["id"]]
            lines.append(f"- [{ex['name_ko']}]({ex['url']}) {e['sets']}세트 × {re.sub(r'\s*회$', '', e['reps'].strip())}회: {e['reason']}")
        lines.append("")
    lines += [f"**유산소**: {routine['cardio_note']}", "", f"**주의**: {routine['caution']}"]
    if removed:
        lines.append(f"\n(도감에 없어 지운 운동: {', '.join(removed)})")
    display(Markdown("\n".join(lines)))


show_routine(PROFILES[0]["file"])

## 다음 단계

- 결과를 `docs/evaluation.md`의 루틴 추천 부분에 정리한다.
- 같은 판정 기준·규칙·필터를 헬스도감 앱(`/routine` 화면)에 옮겨, 사진을 올리면 수치 확인 → 루틴 추천까지 이어지게 한다.